# 11. Seeds, sweeps and Monte-Carlo runs

Every random process of a context (photon and read noise, turbulence screens, Monte-Carlo draws) derives from
**one seed**: the same seed gives the same results. A **sweep** runs the context for a range of values of some
parameters; a **Monte-Carlo run** draws its free parameters from their priors, each draw with its own noise
realization. This notebook studies the tolerance of the focus of a lens with the preset
`monte_carlo.lumos`: a HeNe laser beam of waist 1 mm focused by a 200 mm lens on a camera.

In [ ]:
# On Google Colab only: install LUMOS and fetch the presets used by the examples.
import sys

if "google.colab" in sys.modules:
    %pip install -q "lumos-maxima[catalogs] @ git+https://github.com/VForiel/LUMOS"
    !git clone -q --depth 1 https://github.com/VForiel/LUMOS /content/LUMOS
    !cp -r /content/LUMOS/examples/presets /content/LUMOS/examples/catalog_cache .

In [ ]:
import numpy as np
from astropy import units as u

import lumos
from lumos.plotting import style

style.apply()
ctx = lumos.io.load("presets/monte_carlo.lumos")
print(ctx.seed, ctx.free_parameters())

## A sweep of the focus

A Gaussian beam of waist $w$ focused by a lens of focal length $f$ has a waist
$w_f = \lambda f / (\pi w)$ and a Rayleigh range $z_R = \pi w_f^2 / \lambda$; at a distance $\Delta z$ from
the focus its peak intensity is reduced by $1 / (1 + (\Delta z / z_R)^2)$ (Siegman, *Lasers*, §17.1). The sweep
moves the camera around the focus. It keeps the noise of the context at every point (common random
numbers), so that the curve shows the optics, not the noise.

In [ ]:
distances = np.linspace(190, 210, 21) * u.mm
points = ctx.sweep({"focus.distance": distances}, outputs=["camera.out"])
peaks = np.array([p.outputs["camera.out"].data.value.max() for p in points])

wavelength, waist, focal = 633e-9, 1e-3, 0.2
w_f = wavelength * focal / (np.pi * waist)
z_r = np.pi * w_f**2 / wavelength
dz = distances.to_value(u.m) - focal
expected = peaks.max() / (1 + (dz / z_r) ** 2)
print(f"waist at focus {w_f * 1e6:.1f} um, Rayleigh range {z_r * 1e3:.2f} mm")

fig, ax = style.figure("single")
ax.plot(distances, peaks, "o", label="LUMOS")
ax.plot(distances, expected, label="Gaussian beam")
ax.set_xlabel("distance lens-camera [mm]")
ax.set_ylabel("peak [adu]")
ax.legend();

## A Monte-Carlo run of the tolerance

The focal length of the lens is a free parameter with a normal prior, 200 ± 2 mm: a manufacturing tolerance.
Each draw takes a focal length and its own photon noise (`noise="independent"`, the default). The peaks
spread below the best focus; their mean follows from the same law averaged over the prior.

In [ ]:
draws = ctx.sample(200, outputs=["camera.out"])
focals = np.array([d.params["lens.focal_length"].to_value(u.mm) for d in draws])
peaks = np.array([d.outputs["camera.out"].data.value.max() for d in draws])
best = ctx.sweep({"focus.distance": [200] * u.mm}, outputs=["camera.out"])[0].outputs["camera.out"].data.value.max()
predicted = best * np.mean(1 / (1 + ((focals - 200) * 1e-3 / z_r) ** 2))
print(f"focal lengths {focals.mean():.2f} ± {focals.std(ddof=1):.2f} mm")
print(f"mean peak {peaks.mean():.4g} adu, predicted {predicted:.4g} adu")

fig, axes = style.figure("double", ncols=2, aspect=0.4)
axes[0].hist(focals, bins=20)
axes[0].set_xlabel("focal length [mm]")
axes[1].hist(peaks, bins=20)
axes[1].axvline(predicted, color="C1", label="predicted mean")
axes[1].set_xlabel("peak [adu]")
axes[1].legend();

## Reproducibility and archive

The same seed gives the same draws; another seed gives others. The context, the figures and the data of a
final run are archived next to the notebook.

In [ ]:
again = ctx.sample(5, outputs=["camera.out"])
assert [d.params for d in again] == [d.params for d in draws[:5]]
ctx.seed = 2027
other = ctx.sample(5, outputs=["camera.out"])
assert [d.params for d in other] != [d.params for d in again]

result = ctx.run(ctx.meta["outputs"], archive="monte-carlo")
result.archive.save_figure(fig, "monte_carlo_peaks");